# Laboratorio L7 - Codici TOTP

B.4 - Cybersicurezza e IA

Il notebook implementa l'algoritmo TOTP (RFC 6238), usato dalle app di autenticazione, con la sola libreria standard di Python, e lo verifica con i valori di prova pubblicati nella specifica.

Il segreto usato è di prova: non va usato per account reali.

## 1. Gli ingredienti

- segreto condiviso: sequenza di byte generata dal servizio e memorizzata dall'app (di solito tramite codice QR). Nelle app si scrive in Base32, una codifica che usa lettere maiuscole e cifre da 2 a 7
- contatore: tempo Unix (secondi trascorsi dal 1° gennaio 1970) diviso 30, parte intera
- HMAC-SHA1: hash del contatore calcolato con il segreto come chiave

Moduli usati:

- `hmac`, `hashlib`: calcolo dell'HMAC-SHA1
- `struct.pack(">Q", n)`: converte il numero `n` in 8 byte (formato richiesto dalla specifica)
- `base64.b32decode`: decodifica il segreto dal Base32
- `time.time()`: tempo Unix corrente

In [ ]:
import hmac, hashlib, struct, base64, time

def hotp(segreto, contatore, cifre=6):
    """Codice monouso da un segreto (byte) e da un contatore (intero)."""
    h = hmac.new(segreto, struct.pack(">Q", contatore), hashlib.sha1).digest()
    offset = h[-1] & 0x0F                                  # ultimi 4 bit dell'hash: posizione da cui leggere
    numero = struct.unpack(">I", h[offset:offset + 4])[0] & 0x7FFFFFFF
    return str(numero % 10 ** cifre).zfill(cifre)          # ultime cifre, con zeri iniziali

def totp(segreto, istante=None, intervallo=30, cifre=6):
    if istante is None:
        istante = time.time()
    return hotp(segreto, int(istante // intervallo), cifre)

## 2. Codice corrente con il segreto di prova (esercizio 1)

Rieseguire la cella dopo 30 secondi: il codice cambia.

In [ ]:
SEGRETO_BASE32 = "JBSWY3DPEHPK3PXP"          # segreto di prova, pubblicato in molti esempi
segreto = base64.b32decode(SEGRETO_BASE32)

adesso = time.time()
secondi_rimasti = 30 - int(adesso) % 30
print("codice attuale:", totp(segreto, adesso))
print("valido ancora per circa", secondi_rimasti, "secondi")

## 3. Verifica con i valori ufficiali (esercizio 2)

La specifica RFC 6238 pubblica, per il segreto ASCII `12345678901234567890` e codici di 8 cifre, i codici attesi in alcuni istanti. Se l'implementazione è corretta, i codici coincidono.

`assert condizione, messaggio` interrompe il programma con un errore se la condizione è falsa.

In [ ]:
segreto_rfc = b"12345678901234567890"
VALORI_RFC = {59: "94287082", 1111111109: "07081804", 1111111111: "14050471",
              1234567890: "89005924", 2000000000: "69279037"}

for istante, atteso in VALORI_RFC.items():
    calcolato = totp(segreto_rfc, istante, cifre=8)
    assert calcolato == atteso, f"errore a t={istante}: {calcolato} invece di {atteso}"
    print(f"t = {istante:>10}  codice {calcolato}  corretto")

## 4. Configurare un'app di autenticazione (esercizio 3)

Nelle app di autenticazione si può inserire il segreto a mano (voce simile a "Inserisci una chiave di configurazione"):

- nome dell'account: `Laboratorio B4`
- chiave: il valore di `SEGRETO_BASE32`
- tipo: basato sul tempo

In alternativa, molte app accettano un indirizzo `otpauth://`, che è il contenuto del codice QR mostrato dai servizi:

In [ ]:
print(f"otpauth://totp/Laboratorio%20B4?secret={SEGRETO_BASE32}&issuer=Corso%20B4&digits=6&period=30")
print("codice attuale:", totp(segreto))

## 5. Tolleranza dell'orologio (esercizio 4)

Il servizio accetta il codice dell'intervallo corrente e, per compensare piccoli errori di orologio, anche quello dell'intervallo precedente e di quello successivo.

Modificare `scarto` (in secondi) con i valori 30, 60, 90 e osservare quando il codice viene rifiutato.

In [ ]:
def verifica_codice(segreto, codice, istante, tolleranza=1):
    contatore = int(istante // 30)
    return any(hotp(segreto, contatore + d) == codice for d in range(-tolleranza, tolleranza + 1))

scarto = 30   # secondi di differenza tra l'orologio del telefono e quello del servizio

ora_servizio = time.time()
codice_telefono = totp(segreto, ora_servizio + scarto)
print("codice del telefono:", codice_telefono)
print("accettato dal servizio:", verifica_codice(segreto, codice_telefono, ora_servizio))

## 6. Domande (esercizio 5)

- Chi fotografa il codice QR di configurazione che cosa ottiene?
- Un sito falso non conosce il segreto: come riesce comunque ad accedere a un account protetto da TOTP se la vittima inserisce il codice?
- Quale metodo di autenticazione impedisce questo tipo di attacco, e perché?

Risposte: